# Document Loaders — Padrão Moderno LangChain (v0.2+)

Conversão do notebook original para LCEL (LangChain Expression Language).

**Principais mudanças:**
- `load_qa_chain` + `chain.run()` → LCEL com pipes `|` + `chain.invoke()`
- `OpenAIWhisperParser` importado de `langchain_community`
- Prompt explícito no lugar do prompt embutido do `StuffDocumentsChain`

## Configuração da Chain (LCEL)

Criamos a chain uma única vez e reutilizamos em todos os exemplos abaixo.

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

load_dotenv()

chat = ChatOpenAI(model="gpt-3.5-turbo-0125")

# Prompt equivalente ao chain_type="stuff"
prompt = ChatPromptTemplate.from_template("""
Use o contexto abaixo para responder à pergunta.
Se não souber a resposta, diga que não sabe — não invente.

Contexto:
{context}

Pergunta: {question}
""")

def format_docs(docs):
    """Concatena o conteúdo dos documentos, equivalente ao chain_type='stuff'."""
    return "\n\n".join(doc.page_content for doc in docs)

# Chain LCEL — substitui load_qa_chain(chain_type="stuff")
chain = prompt | chat | StrOutputParser()

def ask(documents, question):
    """Helper: responde uma pergunta dado uma lista de documentos."""
    return chain.invoke({
        "context": format_docs(documents),
        "question": question
    })

---
## Carregamento de PDF

In [2]:
from langchain_community.document_loaders.pdf import PyPDFLoader

arquivo = "../files/apostila.pdf"
loader = PyPDFLoader(arquivo)
documentos = loader.load()

In [3]:
len(documentos)

28

In [4]:
documentos[0]

Document(metadata={'producer': 'Microsoft® Word 2013', 'creator': 'Microsoft® Word 2013', 'creationdate': '2016-05-04T10:06:39-03:00', 'author': 'lucas', 'moddate': '2016-05-04T10:06:39-03:00', 'source': '../files/apostila.pdf', 'total_pages': 28, 'page': 0, 'page_label': '1'}, page_content='INTRODUÇÃO À PROGRAMAÇÃO \nCOM PYTHON \n \n \n \n \n \n \n \nPrograma de Educação Tutorial \nGrupo PET - ADS   \nIFSP -  Câmpus São Carlos')

In [5]:
documentos[5].page_content

'3 \n \n2. Digite o comando sudo apt-get install python3.4 no terminal do GNU/Linux para inicializar \no processo de instalação. \n \n \n \n3. Terminado o download, o interpretador já estará instalado no computador. \n \n \n \n \nb) Instalação do IDLE no Linux \n \nO IDLE é um ambiente in tegrado de desenvolvimento que acompanha a instalação do interpretador \nPython em sistemas operacionais Windows. Para tê -lo disponível em distribuições Linux basta \nseguir as etapas abaixo: \n \n1. Acesse o terminal Linux. \n \n \n \n2. Digite o comando sudo apt-get install idle-python3.4. \n \n \n \n3. Para executá-lo basta digitar no terminal idle-python3.4 &.'

In [6]:
pergunta = "Do que se trata esse documento?"

# Antes:  chain.run(input_documents=documentos[:8], question=pergunta)
# Agora:
resposta = ask(documentos[:8], pergunta)
print(resposta)

Esse documento é uma apostila de Introdução à Programação com Python, desenvolvida pelo grupo PET - ADS do IFSP Campus São Carlos. O documento aborda características da linguagem Python, instalação do interpretador Python, variáveis, strings, números, listas, tuplas, dicionários, bibliotecas, estruturas de decisão, estruturas de repetição, funções e exercícios relacionados a esses temas.


---
## Carregamento de Arquivo CSV

In [7]:
from langchain_community.document_loaders.csv_loader import CSVLoader

arquivo = "../files/imdb_movies.csv"
loader = CSVLoader(arquivo)
documentos = loader.load()

In [8]:
documentos[5]

Document(metadata={'source': '../files/imdb_movies.csv', 'row': 5}, page_content=': 5\nMovie Name: The Godfather: Part II\nYear of Release: (1974)\nWatch Time: 202 min\nMovie Rating: 9.0\nMeatscore of movie: 90\nVotes: 34,709\nGross: $57.30M\nDescription: The early life and career of Vito Corleone in 1920s New York City is portrayed, while his son, Michael, expands and tightens his grip on the family crime syndicate.')

In [9]:
len(documentos)

1000

In [10]:
documentos[80].metadata

{'source': '../files/imdb_movies.csv', 'row': 80}

In [11]:
pergunta = "Qual filme com menor e maior metascore?"

# Antes:  chain.run(input_documents=documentos[:10], question=pergunta)
# Agora:
resposta = ask(documentos[:10], pergunta)
print(resposta)

O filme com menor Metascore é "Jai Bhim" com **** (não especificado) e o filme com maior Metascore é "The Godfather" com 100.


---
## Carregando Vídeos do YouTube

In [12]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

# ATUALIZADO: OpenAIWhisperParser migrou para langchain_community
from langchain_community.document_loaders.parsers.audio import OpenAIWhisperParser

In [13]:
url = "https://www.youtube.com/watch?v=ox3-BEfHHnk&t=16s"
save_dir = "files/youtube"

loader = GenericLoader(
    YoutubeAudioLoader([url], save_dir),
    OpenAIWhisperParser()
)
docs = loader.load()

[youtube] Extracting URL: https://www.youtube.com/watch?v=ox3-BEfHHnk&t=16s
[youtube] ox3-BEfHHnk: Downloading webpage


[youtube] ox3-BEfHHnk: Downloading android vr player API JSON
[info] ox3-BEfHHnk: Downloading 1 format(s): 140
[download] files\youtube\Crie 3 Apps Simples com Python Usando a Biblioteca Gradio.m4a has already been downloaded
[download] 100% of   23.05MiB
[ExtractAudio] Not converting audio files\youtube\Crie 3 Apps Simples com Python Usando a Biblioteca Gradio.m4a; file is already in target format m4a


C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\pydub\utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\pydub\utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\pydub\utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
C:\Users\soste\AppData\Local\pypoetry\Cache\virtualenvs\curso-4-3-2-langchain-crie-agentes-ia-apps-OzPkRWV8-py3.12\Lib\site-packages\pydub\utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(de

Transcribing part 1!
Transcribing part 2!


In [14]:
len(docs)

2

In [15]:
docs[0].page_content[:200]

'até o presente momento nós abordamos os tópicos que são fundamentais ali na programação com a linguagem python principalmente na parte fundamental seja o fundamento seja os módulos orientação objetos '

In [16]:
pergunta = "Faça um resumo breve desse vídeo para mim"

# Antes:  chain.run(input_documents=docs, question=pergunta)
# Agora:
resposta = ask(docs, pergunta)
print(resposta)

No vídeo, o instrutor aborda a utilização da biblioteca Gradio para criar interfaces gráficas em Python. Ele demonstra como criar funções simples para somar números, inverter textos e calcular fatoriais, e em seguida, integra essas funções em interfaces web utilizando o Gradio. O objetivo é facilitar a compreensão dos conceitos fundamentais de Python por meio de exemplos práticos.


---
## Web via URL

In [26]:
import os
os.environ["USER_AGENT"] = "meu-agente/1.0"

from langchain_community.document_loaders.web_base import WebBaseLoader

loader = WebBaseLoader(
    "https://medium.com/ensina-ai/introdu%C3%A7%C3%A3o-a-processamento-de-linguagem-natural-174936c096b",
    header_template={
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "pt-BR,pt;q=0.9",
    }
)
documents = loader.load()
print(documents[0].page_content[:500])

Just a moment...Enable JavaScript and cookies to continue


In [ ]:
len(documents)

In [ ]:
documents[0].page_content[:500]